``NumericalFeature.get_parts`` slices a per-residue ``dict_num`` into the jmd_n / tmd / jmd_c parts that :meth:`CPP.run_num` consumes, returning the ``(df_parts, dict_num_parts)`` pair aligned to ``df_seq``.

In [1]:
import numpy as np
import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

seqs = ["ACDEFGHIKLMNPQRSTVWY" * 3] * 4   # length 60
df_seq = pd.DataFrame({"entry": [f"P{i}" for i in range(4)], "sequence": seqs,
                       "tmd_start": 11, "tmd_stop": 50})
rng = np.random.default_rng(0)
dict_num = {e: rng.random((60, 4)) for e in df_seq["entry"]}

df_parts, dict_num_parts = aa.NumericalFeature().get_parts(df_seq=df_seq, dict_num=dict_num)
df_parts.head()

,tmd,jmd_n_tmd_n,tmd_c_jmd_c
entry,,,
P0,MNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,MNPQRSTVWYACDEFGHIKLMNPQRSTVWY
P1,MNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,MNPQRSTVWYACDEFGHIKLMNPQRSTVWY
P2,MNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,MNPQRSTVWYACDEFGHIKLMNPQRSTVWY
P3,MNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,MNPQRSTVWYACDEFGHIKLMNPQRSTVWY


``dict_num_parts`` carries the per-part tensors aligned to ``df_parts``:

In [2]:
{e: v.shape for e, v in list(dict_num_parts.items())[:2]}

{'tmd': (4, 40, 4), 'jmd_n_tmd_n': (4, 30, 4)}

**Further parameters.** ``NumericalFeature.get_parts`` also accepts: ``list_parts`` — Subset of part names to materialize (e.g; ``all_parts`` — If ``True``, return all available parts; ignored when ``list_parts`` is supplied; ``jmd_n_len`` — Length of JMD-N (>=0); ``jmd_c_len`` — Length of JMD-C (>=0); ``tmd_len`` — Target middle domain (TMD) length for the **anchor-based format** only (a ``sequence`` + ``pos`` ``df_seq``): each 1-based anchor in ``pos`` is exploded into one row with the TMD centered (right-heavy for even ``tmd_len``) on the anchor, and the matching ``dict_num`` tensor is sliced with the same boundaries.

In [3]:
nf = aa.NumericalFeature()
# all_parts=True returns every available part; jmd_n_len / jmd_c_len set the flank lengths
df_parts_all, _ = nf.get_parts(df_seq=df_seq, dict_num=dict_num,
                               all_parts=True, jmd_n_len=10, jmd_c_len=10)
print("all parts:", list(df_parts_all.columns))
# list_parts selects a subset of parts (and overrides all_parts)
df_parts_tmd, _ = nf.get_parts(df_seq=df_seq, dict_num=dict_num, list_parts=["tmd"])
print("subset:", list(df_parts_tmd.columns))
# tmd_len drives the anchor-based format (a 'sequence' + 'pos' df_seq): each 1-based anchor
# is exploded into a TMD of length tmd_len centered on it
df_seq_anchor = pd.DataFrame({"entry": [f"P{i}" for i in range(4)], "sequence": seqs,
                              "pos": [20, 25, 30, 35]})
df_parts_anchor, dict_num_parts_anchor = nf.get_parts(df_seq=df_seq_anchor, dict_num=dict_num,
                                                      tmd_len=10)
aa.display_df(df_parts_anchor, n_rows=10, show_shape=True)

all parts: ['tmd', 'tmd_n', 'tmd_c', 'jmd_n', 'jmd_c', 'tmd_jmd', 'jmd_n_tmd_n', 'tmd_c_jmd_c']
subset: ['tmd']
DataFrame shape: (4, 3)


,tmd,jmd_n_tmd_n,tmd_c_jmd_c
P0_6-35,STVWYACDEF,GHIKLMNPQRSTVWY,ACDEFGHIKLMNPQR
P1_11-40,ACDEFGHIKL,MNPQRSTVWYACDEF,GHIKLMNPQRSTVWY
P2_16-45,GHIKLMNPQR,STVWYACDEFGHIKL,MNPQRSTVWYACDEF
P3_21-50,MNPQRSTVWY,ACDEFGHIKLMNPQR,STVWYACDEFGHIKL
